# 01 — Data Preparation
Download PlantVillage and PlantDoc, create class mappings, generate stratified train/val/test splits, and create multilingual retrieval pairs.

**Run this notebook first before any training notebooks.**

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/farmer_advisory_chatbot'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Project directory on Drive: {PROJECT_DIR}")

In [ ]:
# Setup working directory in Colab local runtime (/content/project)
# If code was zipped on Drive, extract it:
import os, shutil
if os.path.exists(f"{PROJECT_DIR}/code.zip"):
    !unzip -q {PROJECT_DIR}/code.zip -d /content/project
else:
    # Alternatively clone repository or create directory
    os.makedirs("/content/project", exist_ok=True)

%cd /content/project
!pip install -r requirements.txt -q

## 1. Download PlantVillage Dataset
Source: https://github.com/spMohanty/PlantVillage-Dataset (CC0 1.0 Universal License)

In [ ]:
!python data_prep/download_plantvillage.py --real --output_dir data/plantvillage

## 2. Download PlantDoc Dataset
Source: https://github.com/pratikkayal/PlantDoc-Dataset (Research License)

In [ ]:
!python data_prep/download_plantdoc.py --real --output_dir data/plantdoc

## 3. Prepare Stratified Vision Splits
Creates `data/vision_train.csv`, `data/vision_val.csv`, `data/vision_test.csv`, and `data/plantdoc_test.csv` (14 shared classes across 4 crops).

In [ ]:
!python data_prep/prepare_splits.py --pv_dir data/plantvillage --pd_dir data/plantdoc --output_dir data

## 4. Generate Multilingual Retrieval Training Pairs
Generates English, Hindi, and Marathi query pairs from `kb/passages.json` using NLLB-200 translation.

In [ ]:
!python data_prep/generate_retrieval_pairs.py --kb_path kb/passages.json --output_dir data --use_nllb

## 5. Verify and Copy Prepared Data to Drive
Persist all split CSVs to Google Drive so subsequent notebooks (vision, retriever) have direct access.

In [ ]:
import shutil, glob
drive_data = os.path.join(PROJECT_DIR, 'data')
os.makedirs(drive_data, exist_ok=True)

for csv_file in glob.glob('data/*.csv'):
    dest = os.path.join(drive_data, os.path.basename(csv_file))
    shutil.copy2(csv_file, dest)
    print(f"Copied {csv_file} -> {dest}")

shutil.copy2('data/class_mapping.json', os.path.join(drive_data, 'class_mapping.json'))
print("All data splits successfully backed up to Google Drive!")